## Treinamento e avaliação do modelo

##Requirements

In [0]:
%pip install category_encoders optuna scikit-learn pandas optuna-integration xgboost lightgbm catboost


Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install  openpyxl

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, fbeta_score
import optuna
from optuna_integration import OptunaSearchCV
import mlflow
from processing import preprocessing
from model import build_model
from metrics import metrics_report
from processing import padronizar   
import os
import joblib

 

##Importação e Tratamento de Dados

In [0]:
dados = pd.read_csv("/Workspace/projeto_databricks_churn/Hackathon_Terra_Signal/data/raw/history.csv")                  
dados_inference = pd.read_csv("/Workspace/projeto_databricks_churn/Hackathon_Terra_Signal/data/raw/inference.csv")                  
dados.head()    
                      

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,3482.85,No,I have been a customer with this internet prov...,6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.90,2405.05,Yes,I've been a customer with this company for ove...,7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 2...,3628
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet prov...,7851
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet ser...,1691


In [0]:
dados = padronizar(dados)
dados_inference = padronizar(dados_inference)
dados.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,1,1,52.0,1,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,1,Electronic check,68.75,3482.85,0,I have been a customer with this internet prov...,6532
1,6289-CPNLD,Male,0,1,1,33.0,1,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,1,Mailed check,73.90,2405.05,1,I've been a customer with this company for ove...,7634
2,2682-KEVRP,Female,1,0,0,22.0,1,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,1,Mailed check,20.05,417.00,0,I have been a customer with this company for 2...,3628
3,5697-GOMBF,Female,1,1,1,28.0,0,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,1,Electronic check,35.90,973.65,0,I have been a customer with this internet prov...,7851
4,9717-QEBGU,Male,0,0,0,2.0,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.30,144.00,0,I have been using the fiber optic internet ser...,1691


## Divisao de Dados de Treino e teste

In [0]:
X = dados.drop(columns=["Churn", "CustomerFeedback", "customerID"])
y = dados['Churn']

xcat = dados.copy()\
            .select_dtypes(include=['object'])\
                .drop(columns=["customerID", "CustomerFeedback"])\
                   .columns.to_list()

xnum = dados.copy()\
            .select_dtypes(include=['int64', 'float64'])\
                .drop(columns=["Churn"])\
                    .columns.to_list()

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


##Definição de Hiperparametros

In [0]:
models_params = {'random_forest': {'model__n_estimators': optuna.distributions.IntDistribution(100, 1000),
                                   'model__max_features': optuna.distributions.FloatDistribution(0.15, 1.0),
                                   'model__max_depth': optuna.distributions.IntDistribution(3, 20), 
                                   'model__min_samples_split': optuna.distributions.IntDistribution(50, 200),
                                   'model__min_samples_leaf': optuna.distributions.IntDistribution(25, 100)
                                  },                 
                    
                 'lgb':{'model__learning_rate': optuna.distributions.FloatDistribution(0.01, 0.15),  
                        'model__num_leaves': optuna.distributions.IntDistribution(8, 25),       	 
                        'model__max_depth': optuna.distributions.IntDistribution(2,8),       	 
                        'model__min_data_in_leaf': optuna.distributions.IntDistribution(50, 550),   
                        'model__bagging_fraction': optuna.distributions.FloatDistribution(0.3, 1.0),  
                        'model__feature_fraction': optuna.distributions.FloatDistribution(0.3, 1.0),  
                        'model__lambda_l2': optuna.distributions.IntDistribution(1, 9) 
                        },

                 'catboost': {'model__learning_rate': optuna.distributions.FloatDistribution(0.01, 0.05),
                              'model__depth': optuna.distributions.IntDistribution(2, 10),  
                              'model__iterations': optuna.distributions.IntDistribution(100, 1000),  
                              'model__subsample': optuna.distributions.FloatDistribution(0.3, 1.0),  
                              'model__l2_leaf_reg': optuna.distributions.IntDistribution(1, 5),  
                              'model__random_strength': optuna.distributions.FloatDistribution(0.1, 2.0) 
                              },
                                  
                 'xgb': {'model__learning_rate':optuna.distributions.FloatDistribution(    0.01, 0.15, log=True),
                         'model__max_depth': optuna.distributions.IntDistribution(2, 5),
                         'model__subsample': optuna.distributions.FloatDistribution(0.6, 1.0),
                         'model__colsample_bytree':optuna.distributions.FloatDistribution(0.4, 0.8),
                         'model__gamma':optuna.distributions.FloatDistribution(1.0, 8.0),
                         'model__reg_lambda':optuna.distributions.FloatDistribution(5.0, 30.0, log=True),
                         'model__reg_alpha':optuna.distributions.FloatDistribution(1e-5, 1.0, log=True),
                         'model__min_child_weight':optuna.distributions.FloatDistribution(3.0, 15.0, log=True),
                         'model__scale_pos_weight':optuna.distributions.FloatDistribution(1.0, 3.0),
                         'model__n_estimators':optuna.distributions.IntDistribution(300, 1500)
                         }
                }


## Tuning do Modelo

In [0]:
mlflow.set_experiment("/Users/lucasfellipe8888@gmail.com/churn_experiment_terra_signal")
#%%
for model in list(models_params.keys()):
    # itera sobre o dicionario de modelos e parametros iniciando uma execucao mlflow para cada modelo
    with mlflow.start_run(run_name=model):
        pipe_model = Pipeline(steps=[('preprocessing', preprocessing(xnum=xnum, xcat=xcat)),
                                     ('model', build_model(model_name=model))])
        
        # usa o conjunto de parametros específico para o modelo atual
        param_dist = models_params.get(model)

        search = OptunaSearchCV(pipe_model, 
                                param_distributions=param_dist,
                                n_trials=500,
                                cv=RepeatedStratifiedKFold(n_splits=5,                                                     
                                                   random_state=42,
                                                   n_repeats=3),
                                scoring="average_precision", # average_precision otimiza precisao e recall(dar atençao pra precisao evitando sacrificar excessivamente)                                
                                return_train_score=True,
                                n_jobs=-1,
                                random_state=42)

        search.fit(xtrain, ytrain)

        ytrain_pred_prob = search.predict_proba(xtrain)[:, 1]
        ytest_pred_prob = search.predict_proba(xtest)[:, 1]

        report_metrics = metrics_report(ytrain_true=ytrain, ytrain_pred_prob=ytrain_pred_prob, 
                                        ytest_true=ytest, ytest_pred_prob=ytest_pred_prob, 
                                        model=search, threshold=0.5)        
        
        mlflow.sklearn.log_model(
            search.best_estimator_, "model")  
        mlflow.log_metrics(report_metrics)
        mlflow.log_params(search.best_params_)
        

- Salvando e serializando final

In [0]:
run_id = "25534c7acbc84a2fbae62248f90f79bd" 
model_uri = f"runs:/{run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

output_dir = "/Workspace/projeto_databricks_churn/Hackathon_Terra_Signal/models"
os.makedirs(output_dir, exist_ok=True)  

model_path = os.path.join(output_dir, "XGBoosting_final_model.joblib")

joblib.dump(loaded_model, model_path)

print(f"Modelo salvo com sucesso em: {model_path}")


Modelo salvo com sucesso em: /Workspace/projeto_databricks_churn/Hackathon_Terra_Signal/models/XGBoosting_final_model.joblib


##Analise de Cenarios de Lucro e Metricas  

- carregando modelo final e prevendo churn

In [0]:

# carregando modelo e prevendo churn
run_id = "25534c7acbc84a2fbae62248f90f79bd" # aqui é o source run id ou model id
model_uri = f"runs:/{run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

inference_tabela_predicao = dados_inference.copy()
inference_probabilities = loaded_model.predict_proba(inference_tabela_predicao)[:, 1]
inference_tabela_predicao['probabilidade_churn'] = inference_probabilities
inference_tabela_predicao['predict_churn'] = inference_tabela_predicao['probabilidade_churn'].apply(lambda x: 1 if x >= 0.65 else 0)

history_tabela_predicao = dados.copy()
history_probabilities = loaded_model.predict_proba(history_tabela_predicao)[:, 1]
history_tabela_predicao['probabilidade_churn'] = history_probabilities

history_tabela_predicao['predict_churn'] = history_tabela_predicao['probabilidade_churn'].apply(lambda x: 1 if x >= 0.65 else 0)


- Calculando metricas financeiras e de qualidade do modelo

In [0]:
import numpy as np
import pandas as pd

ytrain_pred_prob = loaded_model.predict_proba(xtrain)[:,1]
ytest_pred_prob = loaded_model.predict_proba(xtest)[:,1]

thresholds= [0.4, 0.5, 0.65, 0.7, 0.8, 0.9]# np.arange(0.60, 0.7, 0.01)
results = []

for threshold in thresholds:
  report_metrics = metrics_report(
      ytrain_true=ytrain,
      ytrain_pred_prob=ytrain_pred_prob,
      ytest_true=ytest,
      ytest_pred_prob=ytest_pred_prob,
      model=loaded_model,
      threshold=threshold
  )

  perda_evitada_com_modelo = np.where(
    (history_tabela_predicao['Churn'] == 1) & (history_tabela_predicao['probabilidade_churn'] >= threshold),
    history_tabela_predicao['MonthlyCharges'], 0,
  ).sum()

  perda_esperada_sem_modelo = np.where(
                                history_tabela_predicao['Churn'] == 1,
                                history_tabela_predicao['MonthlyCharges'],   0,)\
                                    .sum()

  current_result = {"threshold": threshold,
                    "porcentagem_da_perda_evidada": round(perda_evitada_com_modelo / perda_esperada_sem_modelo, 3),
                    "perda_evitada_com_modelo_1mes": round(perda_evitada_com_modelo, 2),
                    "perda_esperada_sem_modelo_1mes": round(perda_esperada_sem_modelo, 2),                    
                    "perda_evitada_com_modelo_6meses": round((perda_evitada_com_modelo * 6), 2),
                    "perda_esperada_sem_modelo_6meses": round((perda_esperada_sem_modelo * 6), 2),
                    "perda_evitada_com_modelo_12meses": round((perda_evitada_com_modelo * 12), 2),
                    "perda_esperada_sem_modelo_12meses": round((perda_esperada_sem_modelo * 12), 2)}    
  current_result.update(report_metrics)

  results.append(current_result)

df_results = pd.DataFrame(results)
display(df_results)                    


threshold,porcentagem_da_perda_evidada,perda_evitada_com_modelo_1mes,perda_esperada_sem_modelo_1mes,perda_evitada_com_modelo_6meses,perda_esperada_sem_modelo_6meses,perda_evitada_com_modelo_12meses,perda_esperada_sem_modelo_12meses,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test,roc_auc_train,roc_auc_test,pr_auc_train,pr_auc_test
0.4,0.851,95557.95,112242.9,573347.7,673457.4,1146695.4,1346914.8,0.518,0.499,0.838,0.816,0.64,0.62,0.858,0.846,0.692,0.652
0.5,0.767,86099.25,112242.9,516595.5,673457.4,1033191.0,1346914.8,0.579,0.563,0.734,0.725,0.647,0.634,0.858,0.846,0.692,0.652
0.65,0.603,67647.75,112242.9,405886.5,673457.4,811773.0,1346914.8,0.676,0.653,0.572,0.572,0.619,0.61,0.858,0.846,0.692,0.652
0.7,0.503,56410.1,112242.9,338460.6,673457.4,676921.2,1346914.8,0.71,0.693,0.476,0.47,0.57,0.56,0.858,0.846,0.692,0.652
0.8,0.275,30922.45,112242.9,185534.7,673457.4,371069.4,1346914.8,0.828,0.75,0.256,0.253,0.392,0.378,0.858,0.846,0.692,0.652
0.9,0.03,3345.7,112242.9,20074.2,673457.4,40148.4,1346914.8,0.96,0.929,0.023,0.029,0.045,0.056,0.858,0.846,0.692,0.652
